# Workflow
- Get user input of a job description in plain text
- Put the job description in a user prompt; scrape the job description link if applicable 
- Use the deep research prompts to generate job interview prep plans
- Review and edit the plan with another LLM as a judge
- Create notebooks for interview prep and practice

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## Get the Job Description

In [ ]:
# this will take user from a floating window from vscode
job_description = input("Please enter the job description in plain text: ").strip()

In [ ]:
display(Markdown(job_description))

In [ ]:
JOB_TITLE = "Principal Data Scientist"
CODING_CHALLENGE_AREA= "Python and SQL"
CASE_STUDY_FOCUS_TOPIC = "causal inference"
DAILY_TIME_COMMITMENT_IN_HOURS = 4
OVERALL_TIME_COMMITMENT_IN_DAYS = 14

In [ ]:
initial_system_prompt = f"""
Generate a comprehensive, {OVERALL_TIME_COMMITMENT_IN_DAYS}-day interview preparation curriculum for a {JOB_TITLE} job seeker, customized to the user-provided job description. Leverage up-to-date web search to ensure coverage of the most likely interview topics for the specific job title, skills, company, and sector. Structure content into {DAILY_TIME_COMMITMENT_IN_HOURS} hours of study per day. The curriculum’s aim is to coach and holistically prepare the candidate for all interview components.

- For each day, provide:
  - Targeted {CODING_CHALLENGE_AREA} coding challenges
  - Review material on key topics (e.g., statistics, machine learning, generative AI, agentic AI), drawing from job description and web search for added required topics
  - One tailored case study daily emphasizing {CASE_STUDY_FOCUS_TOPIC}, clearly linked to the role; ensure the case uses a publicly available dataset, and include code to access and load the dataset
  - Any additional study resources or exercises that are commonly recommended (per current web search) for interviews matching the job and company context
  - All materials for each day should be structured as a single Jupyter notebook (.ipynb) per day

- Reason step-by-step:
  - First, analyze the job description for required skills, tools, and domain focus
  - Next, survey authoritative web resources for up-to-date interview patterns for the given role/company
  - Carefully select or create content for each daily module, matching the most frequently assessed skills
  - For each day, provide clear objectives, actual coding/content segments, and brief solution keys (for challenges and case studies)
- Remain persistent: Continue through all {OVERALL_TIME_COMMITMENT_IN_DAYS} days unless job description suggests a radically different study period.
- Chain of thought: Think step-by-step for tailoring, sequencing, inclusion of relevant skills, and matching difficulty to the intended candidate level.
- For all coding, provide high-quality, realistic problems and code with explanations as markdown cells inside the notebook structure.
- Maintain a focus on {CASE_STUDY_FOCUS_TOPIC} within case studies, ensuring practical application for the likely interview context.
- Output: One high-level detailed day-by-day study plan as an ordered list, with summary section per day. For each day, outline:
  - "Daily Objective"
  - "Python and SQL Challenges"
  - "Review Topics"
  - "Case Study" (including dataset access code)
  - "Additional Resources/Exercises"
  
- Output format: return one ordered markdown list for the full {OVERALL_TIME_COMMITMENT_IN_DAYS}-day study plan, with indented subsections for each day as specified. Do not wrap output in code blocks; use markdown formatting only.
- Response style: Provide a concise, structured, and clear outline for each day, with placeholders for actual coding problems and case studies. Use markdown headers and bullet points for clarity.

#### Example

Below is a brief sample for illustration; actual outputs should be more detailed, with realistic challenge and case study placeholders.

Day 1:
  - Daily Objective: Review core Python fundamentals and SQL SELECT queries. Introduce causal inference via simple A/B testing.
  - Python and SQL Challenges: [2 Python coding questions on lists and dictionaries], [1 SQL SELECT problem on a public dataset]
  - Review Topics: Python basics, core SQL syntax, basic statistics (mean, median, mode)
  - Case Study: A/B test analysis using [OpenPaymentsDataset]; provide code to load dataset from Kaggle
  - Additional Resources/Exercises: [Link to LeetCode SQL easy playlist], [Blog post on causal inference 101]

[...]

**Important Considerations:**
- Ensure case studies, review topics, and additional exercises are explicitly relevant to BOTH the job description and current interview expectations, as determined by recent reliable sources.
- Prioritize breadth for coverage, and depth for any key skills repeatedly mentioned in the job requirements or by current industry guides.
- Each day’s outline should be sufficiently detailed to guide the creation of a real .ipynb file with all included components but do not create the actual files.

---

**Reminders:**  
The prompt’s purpose is to generate a highly structured and tailored {OVERALL_TIME_COMMITMENT_IN_DAYS}-day interview prep curriculum for data science job seekers, using job-descriptions and web search for real-world relevance.  
Ensure each day’s outline covers coding, theory, a causal inference-focused case study with public data/code, and extra resources, all as a single notebook per day.  
Reason step-by-step; conclusion comes at the end of each daily section.

"""

In [ ]:
initial_user_prompt = f"""
Please generate a interview preparation curriculum for this job description: {job_description}
"""

In [ ]:
initial_messages =[{"role": "system", "content": initial_system_prompt.strip()},
                   {"role": "user", "content": initial_user_prompt.strip()}]

In [ ]:
initial_plan = openai.chat.completions.create(model="gpt-6-astra", messages=initial_messages, reasoning_effort="high")
display(Markdown(initial_plan.choices[0].message.content))

In [ ]:
with open("initial_plan.json", "w", encoding="utf-8") as file:
    file.write(initial_plan.model_dump_json(indent=2))

In [ ]:
critique_prompt = f"""
You are a highly skilled data science interview coach. Read the following 14-day interview preparation curriculum and the job description provided. 
Critically evaluate the curriculum for completeness, relevance, and alignment with the job description. 
Identify any gaps in topics, skills, or case studies that should be addressed to ensure the candidate is fully prepared for the interview process. 
Edit the curriculum to fill in any gaps, and provide a revised version that is improved based on your evaluation.
Job Description: {job_description}
Curriculum: {initial_plan.choices[0].message.content}
"""
critique_messages =[{"role": "user", "content": critique_prompt.strip()}]

In [ ]:
revised_plan = anthropic.chat.completions.create(model="claude-fable-5-2", messages=critique_messages, reasoning_effort="high")
display(Markdown(revised_plan.choices[0].message.content))

In [ ]:
with open("revised_plan.json", "w", encoding="utf-8") as file:
    file.write(revised_plan.model_dump_json(indent=2))

In [ ]:
critique_prompt_2 = f"""
You are a highly skilled data science interview coach. Read the following revised 14-day interview preparation curriculum and the job description provided. 
Critically evaluate the curriculum for completeness, relevance, and alignment with the job description. 
Identify any gaps in topics, skills, or case studies that should be addressed to ensure the candidate is fully prepared for the interview process. 
Edit the curriculum to fill in any gaps, and provide a revised version that is improved based on your evaluation.
Job Description: {job_description}
Curriculum: {revised_plan.choices[0].message.content}
"""
critique_messages_2 =[{"role": "user", "content": critique_prompt_2.strip()}]

In [ ]:
final_plan = openrouter.chat.completions.create(model="google/gemini-4-carbon", messages=critique_messages_2, reasoning_effort="high")
display(Markdown(final_plan.choices[0].message.content))

In [ ]:
display(Markdown(final_plan.choices[0].message.content))

In [ ]:
with open("final_plan.json", "w", encoding="utf-8") as file:
    file.write(final_plan.model_dump_json(indent=2))